In [33]:
import os
import urlrequest
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

### Encode text with byte pair encoding (BPE)
BPE has a vocabulary size of 50,257.

In [3]:
tokenizer = tiktoken.get_encoding('gpt2')

In [15]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces "
     "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


### Decode text

#### BPE can handle out-of-vocabulary words such as 'someunknownPlace'

In [16]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [17]:
ints = tokenizer.encode("someunknownPlace")
print(ints)
strings = tokenizer.decode(ints)
print(strings)

[11246, 34680, 27271]
someunknownPlace


### Encode "The Verdict" book and show a sample of inputs and targets.

In [18]:
with open("the-verdict.txt") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


Remove the first 50 tokens for demonstration purposes, as it results in a slightly more interesting text passage in the next steps.

In [19]:
enc_sample = enc_text[50:]

Create input target pairs.\
x contains the input tokens.\
y contains the targets.

In [23]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


Show next word predictions.

In [27]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [31]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


### Load the inputs and targets as pytorch tensors.

In [41]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256,
                          stride=128, shuffle=True, drop_last=True,
                          num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                            drop_last= drop_last, num_workers=num_workers)
    return dataloader

Test the data loader.

In [43]:
with open("the-verdict.txt") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)

for i in range(2):
    batch = next(data_iter)
    print(batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


### Batch sizes
Smaller batch sizes use less memory, but lead to more noisy model updates.

In machine learning, batch size is a hyperparameter that defines the number of training samples processed before the model's internal parameters (weights) are updated. It is a critical "lever" that balances training speed, memory consumption, and the model's ability to generalize to new data.

Increase the batch size of 8.\
Increase the stride to 4 to avoid overlap between the batches which could lead to increased overfitting.

In [74]:
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)

inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("Targets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### Create token embeddings from the token ids

#### A simple example
6 word vocabulary\
4 sample tokens\
Create a weight matrix of random values that can be optimized via backpropagation.

In [80]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print("Create a weight matrix of random values with "
      f"{vocab_size} rows of {output_dim} dimensions (cols).\n")

print(f"Weight matrix for a vocab of size {vocab_size}:")
print(embedding_layer.weight, "\n")

input_ids = torch.tensor([2, 3, 5, 1])
print(len(input_ids), "token embeddings:")
print(embedding_layer(input_ids))

Create a weight matrix of random values with 6 rows of 3 dimensions (cols).

Weight matrix for a vocab of size 6:
Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True) 

4 token embeddings:
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


### Encoding word positions

Absolute positional embeddings:\
A unique embedding is adding to the token embedding to convey it's location.\
The model learns the exact token positioning.

Relative postional embeddings:\
The relative position or distance between tokens.
The model learns how far apart tokens are.

In [73]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


GPT models use absolute positioning, so create another embedded layer for it.\
Each position is a row vector of output_dim cols that can be added to the token_embeddings to get the input_embeddings.

In [75]:
context_length = max_length  # number of token positions
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])
